# Holzinger–Swineford plasmode simulation

Board workflow:
1. R: fit lavaan CFA on standardized Holzinger `x1..x9` → export `Λ, Φ, Ω`
2–4. Python: build `Σ_X = ΛΦΛᵀ+Ω`, joint `(X,Z)` law, run AIPW sim
5. Summaries: rel bias, 95% coverage, var ratio, PSD / Heywood rates

This notebook owns the **Python** steps. Run the R export once first.

## 0. Setup

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts" / "simulation.py").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT))

from metrics import summarize_run
from misspec import available_scenarios
from plasmode import DEFAULT_PARAMS_DIR, load_plasmode_joint
from simulation import RunConfig, run_simulation

ROOT, DEFAULT_PARAMS_DIR

## 1. R export (once)

Writes CSVs under `params/plasmode_holzinger/`.
Skip if those files already exist.

In [ ]:
import subprocess

params_dir = DEFAULT_PARAMS_DIR
needed = ["lambda.csv", "phi.csv", "omega.csv", "meta.json"]
missing = [f for f in needed if not (params_dir / f).exists()]

if missing:
    r_script = ROOT / "scripts" / "R" / "plasmode_holzinger_fa.R"
    cmd = ["Rscript", str(r_script), str(params_dir)]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(ROOT), check=True)
else:
    print("FA params already present at", params_dir)

list(params_dir.glob("*"))

## 2–3. Load FA params and build `Σ`

In [ ]:
# Legacy p=6 used cov_XZ=0.64; for Holzinger p=9 that is non-PD.
# Default: 0.9 * max equal cov that keeps joint Sigma PD (~0.19 here).
plas = load_plasmode_joint(
    params_dir,
    cov_XZ=None,
    var_Z=1.0,
    mean_value=2.0,
)

print("p =", plas["p"], "n_factors =", plas["n_factors"])
print("variables:", plas["variables"])
print("factors:", plas["factors"])
print("cov_XZ =", plas["cov_XZ"], "(max PD =", plas["cov_XZ_max"], ")")
print("Sigma shape (X,Z):", plas["Sigma"].shape)
print("min eig Sigma_X:", np.linalg.eigvalsh(plas["Sigma_X"]).min())
print("min eig Sigma:", np.linalg.eigvalsh(plas["Sigma"]).min())

display(pd.DataFrame(plas["Lambda"], index=plas["variables"], columns=plas["factors"]))
display(pd.DataFrame(plas["Phi"], index=plas["factors"], columns=plas["factors"]))
display(pd.Series(np.diag(plas["Omega"]), index=plas["variables"], name="uniqueness"))

## 4–5. Simulation knobs

- `n = [250, 500, 1000]` (each `n` → its own `runs/..._n{N}` folder)
- scenarios = all previous misspec conditions
- outcomes written by `summarize_run` after each run

Start with a small `reps` smoke; bump to `1000` for the real study (slow: sandwich FD).

In [ ]:
# Board: n in {250, 500, 1000}, all previous data conditions
n = [250, 500, 1000]
scenarios = list(available_scenarios())  # correct, outcome_mis, propensity_mis, both_mis

reps = 2          # smoke; set to 1000 for full
seed = 20261123
save_data = False

scenarios, n, reps

In [ ]:
run_dirs = []

for scenario in scenarios:
    cfg = RunConfig(
        n=n,
        reps=reps,
        Sigma=plas["Sigma"],
        mean=plas["mean"],
        n_z=1,
        seed=seed,
        scenario=scenario,
        methods=("aipw",),
        propensity_intercept=-0.6,
        propensity_coef=1.0,
        use_true_pi=False,
        save_data=save_data,
        cfa_n_factors=plas["cfa_n_factors"],  # 3 for Holzinger
        run_id=f"holzinger_{scenario}",
    )
    out = run_simulation(cfg)
    if not isinstance(out, list):
        out = [out]
    run_dirs.extend(out)
    for d in out:
        print(scenario, "->", d)

run_dirs

## Step 5 outcomes

`run_simulation` already wrote summaries. Reload / display them here.

In [ ]:
rows = []
for run_dir in run_dirs:
    summary = summarize_run(run_dir, write=True)
    overall = summary["overall_table"].copy()
    params = json.loads((run_dir / "params.json").read_text())
    overall.insert(0, "run_id", run_dir.name)
    overall.insert(1, "n", int(params["n"]))
    overall.insert(2, "scenario", params.get("scenario"))
    rows.append(overall)

results = pd.concat(rows, ignore_index=True)

# Board metrics
cols = [
    "run_id", "n", "scenario", "method",
    "cov_rel_bias",
    "cov_mean_coverage95",
    "cov_mean_var_ratio",
    "psd_rate",
    "heywood_rate",
    "cov_var_trace",
    "cov_mse",
]
cols = [c for c in cols if c in results.columns]
board = results[cols].copy()
# prop non-PD = 1 - PSD rate
if "psd_rate" in board.columns:
    board.insert(board.columns.get_loc("psd_rate") + 1, "prop_non_pd", 1.0 - board["psd_rate"])
board

In [ ]:
# Optional: per-entry var_ratio / coverage for one run
example = run_dirs[0]
entry_path = example / "output" / "summary_aipw_covariance_by_entry.csv"
entry = pd.read_csv(entry_path) if entry_path.exists() else None
print(example.name)
entry

In [ ]:
# Optional: CFA parameter summary for one run
cfa_path = example / "output" / "summary_aipw_cfa_by_param.csv"
cfa = pd.read_csv(cfa_path) if cfa_path.exists() else None
cfa